# Experiment 3 — the prompt ladder at a sample size that can resolve it

Two general checkpoints, three prompt conditions plus the trigger, two difficulty bands, *n* = 100
paired items. Experiments 1 and 2 established the shape at *n* = 20, where only effects above
~25 points were resolvable; this measures the ones that matter at ~8.

Three things it decides, all by rule:

1. **Which difficulty band** leaves the models enough headroom for prompting to move them. The
   correlation between bare accuracy and few-shot gain was **-0.79** at *n* = 20, so the band is the
   variable that matters most.
2. **Whether "explain step by step" does anything** to a model that already reasons unprompted.
   Measured null at *n* = 20; at *n* = 100 a null becomes a statement rather than a shrug.
3. **What suppression costs**, which is the page's headline and the only effect large enough to
   have been safe at *n* = 20.

Presentation tables are emitted as `tables.md` alongside `derived.json`. Writes `out/experiment_3/`.

## How to run this

| models | item sets | conditions | items | generations |
|---|---|---|---|---|
| 2 | 2 | 4 | 100 | **1,600** |

Both checkpoints are bf16 and small, so this is roughly 2 hours with no downloads.

`n = 100` rather than 150 because MATH-500 holds only 105 level-3 problems, and the bands have to
match for the difficulty comparison to mean anything. At the 15% paired discordance measured in
experiment 2, 100 items resolve about ±8 points, which clears every effect in play.

To add the level-3 band, put `"math500_l3"` back into `DATASETS_USED` — it costs about 45 minutes.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs.
# ---------------------------------------------------------------------------
N_ITEMS    = 100
FEW_SHOT_K = 2
GEN_CAP    = 2048          # both checkpoints carry 32k context; this is the answer budget
SEED       = 123
DATASETS_USED = ["gsm8k", "math500_l45"]        # add "math500_l3" for the easy band
CONDITIONS    = ["forced_blurt", "bare", "zero_shot_cot", "few_shot"]

# General checkpoints only. The math-tuned models are out: they refuse the blurt
# instruction outright (0% compliance in experiment 2), which makes them a different
# subject -- one for Act 3, not for a page about what prompting does.
# Both are Qwen2.5: the Qwen2 generation strayed from the required answer format on
# 13% of responses against 1-3% here, and format straying is a scoring artefact we do
# not want inside the measurement.
ROSTER = [
    # Qwen2-1.5B is the cross-generation control: same size as Qwen2.5-1.5B, one
    # generation older. If the ladder has the same shape on both, it is not an
    # artefact of one release. Expect it to floor on MATH levels 4-5 -- it scores
    # 20% on level 3 -- in which case that row is a floor check, not a result.
    dict(id="Qwen/Qwen2-1.5B-Instruct",   tag="Qwen2-1.5B",   quant="bf16", no_think=False),
    dict(id="Qwen/Qwen2.5-1.5B-Instruct", tag="Qwen2.5-1.5B", quant="bf16", no_think=False),
    dict(id="Qwen/Qwen2.5-3B-Instruct",   tag="Qwen2.5-3B",   quant="bf16", no_think=False),
    # 8-bit because 15.24 GB of bf16 weights do not fit in 16 GB. Quantisation is a
    # constant within a model, so it cannot affect the prompt contrasts this page
    # claims -- only cross-model accuracy comparison, which it does not.
    dict(id="Qwen/Qwen2.5-7B-Instruct",   tag="Qwen2.5-7B",   quant="8bit", no_think=False,
         # 90% bare on GSM8K in experiment 2, and GSM8K has no harder band to
         # escape to, so that pass would measure a ceiling and cost 40 minutes.
         skip_datasets=["gsm8k"]),
]

PLANNED = sum(N_ITEMS * len(CONDITIONS)
              for e in ROSTER for ds in DATASETS_USED
              if ds not in e.get("skip_datasets", []))
print(f"{len(ROSTER)} models x {len(DATASETS_USED)} item sets x {len(CONDITIONS)} conditions "
      f"x {N_ITEMS} items = {PLANNED} generations")
for e in ROSTER:
    if e.get("skip_datasets"):
        print(f"   {e['tag']}: skipping {e['skip_datasets']}")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch
from tqdm.auto import tqdm

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/experiment_3"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
TABLES_PATH = OUT_DIR / "tables.md"

mean, median, wilson = harness.mean, harness.median, harness.wilson
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The item sets

Three bands available, two used by default: **GSM8K**, **MATH-500 levels 4-5**, and MATH-500
level 3. The same seed and the same items for every model, so all comparisons are paired.

**The exemplars are repaired here.** GSM8K's reference solutions end in prose with no `\boxed{}`,
while ch03's stem instructs the model to produce one. Few-shot prompting works by imitation, so we
were demonstrating one format and instructing another — and in experiment 2 that flipped 55% of one
model's responses out of the required format. The final answer is now appended as `\boxed{...}` to
any exemplar that lacks it, so the demonstration and the instruction agree.

In [ ]:
GSM8K_URL = ("https://raw.githubusercontent.com/openai/grade-school-math/"
             "master/grade_school_math/data/test.jsonl")


def load_gsm8k(path=DATA_DIR / "gsm8k_test.jsonl"):
    if not path.exists():
        path.write_text(requests.get(GSM8K_URL, timeout=60).text, encoding="utf-8")
    rows = []
    for i, line in enumerate(path.read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        r = json.loads(line)
        body, _, final = r["answer"].rpartition("####")
        rows.append({"unique_id": f"gsm8k/{i}", "problem": r["question"],
                     "answer": final.strip().replace(",", ""),
                     "solution": re.sub(r"<<.*?>>", "", body).strip(),
                     "level": 0, "subject": "grade school"})
    return rows


def box_exemplar(row):
    """Make the demonstration agree with the instruction: every exemplar ends in
    \\boxed{answer}. Without this the few-shot condition measures an
    instruction-versus-demonstration conflict rather than the effect of an example."""
    sol = row["solution"].strip()
    if "\\boxed" in sol:
        return row
    return {**row, "solution": f"{sol}\nThe final answer is $\\boxed{{{row['answer']}}}$."}


def draw(rows, n, seed_offset):
    rows = sorted(rows, key=lambda r: r["unique_id"])
    random.Random(SEED + seed_offset).shuffle(rows)
    return rows[:n], [box_exemplar(r) for r in rows[n:n + FEW_SHOT_K]]


math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
BANDS = {"gsm8k": load_gsm8k(),
         "math500_l3": [r for r in math500 if int(r["level"]) == 3],
         "math500_l45": [r for r in math500 if int(r["level"]) in (4, 5)]}

DATASETS = {}
for name in DATASETS_USED:
    rows = BANDS[name]
    assert len(rows) >= N_ITEMS + FEW_SHOT_K, \
        f"{name} holds {len(rows)} problems, too few for n={N_ITEMS} plus {FEW_SHOT_K} exemplars"
    items, shots = draw(rows, N_ITEMS, 0)
    DATASETS[name] = {"items": items, "exemplars": shots,
                      "hash": harness.sha256_of(r["unique_id"] for r in items)}
    print(f"{name:<13} {len(items)} scored + {len(shots)} exemplars   of {len(rows)} available   "
          f"sha256 {DATASETS[name]['hash'][:12]}")

print("\nexemplar 1, as the model will see it:\n")
print(DATASETS[DATASETS_USED[0]]["exemplars"][0]["solution"][:400])
assert all("\\boxed" in e["solution"] for d in DATASETS.values() for e in d["exemplars"]), \
    "an exemplar still lacks the answer format the stem asks for"
print("\nall exemplars end in the answer format the stem asks for")

## Grading

Inherited from ch03 unchanged, serving both datasets.

In [ ]:
CHECKS = [
    (r"The answer is \boxed{18}.",           "18",   True),    # MATH shape
    (r"She makes $18 every day.",            "18",   True),    # GSM8K shape, no box
    (r"\boxed{\frac{1}{2}}",                 "0.5",  True),
    (r"\boxed{1,024}",                       "1024", True),
    (r"I am not sure.",                      "18",   False),
    (r"Step 1: 16-3=13. Step 2: 13-4=9. So \boxed{18}.", "18", True),
]
rows = [(t, g, e, bool(grade_answer(extract_final_candidate(t), g))) for t, g, e in CHECKS]
bad = [r for r in rows if r[2] != r[3]]
if bad:
    for t, g, e, got in rows:
        print(f"{'ok ' if e == got else 'FAIL'} {extract_final_candidate(t)!r:<28} vs {g!r:<8} {got}")
    raise AssertionError(f"{len(bad)} grader check(s) behaved unexpectedly")
print(f"grader preflight: {len(rows)}/{len(rows)} as expected, both dataset shapes")

## The four conditions

| condition | added after ch03's stem |
|---|---|
| `forced_blurt` | "Give only the final answer. No explanation, no working, no other text." |
| `bare` | nothing |
| `zero_shot_cot` | "Explain step by step." |
| `few_shot` | k reference solutions prepended |

In [ ]:
BLURT_SUFFIX = "\n\nGive only the final answer. No explanation, no working, no other text."
COT_SUFFIX   = "\n\nExplain step by step."


def build_prompt(condition, question, exemplars):
    stem = render_prompt(question)
    if condition == "bare":
        return stem
    if condition == "forced_blurt":
        return stem + BLURT_SUFFIX
    if condition == "zero_shot_cot":
        return stem + COT_SUFFIX
    shots = "".join(render_prompt(e["problem"]) + "\n" + e["solution"].strip() + "\n\n"
                    for e in exemplars[:FEW_SHOT_K])
    return shots + stem


MIN_STEP_CHARS, MIN_CHAIN_STEPS, COMPLIES_UNDER = 12, 2, 40


def _raw_spans(text):
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def n_steps(text):
    text = re.sub(r"^\s*<think>\s*</think>\s*", "", text.strip())
    raw, out = _raw_spans(text), []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        mergeable = (len(piece) < MIN_STEP_CHARS and not re.search(r"\d", piece)
                     and k != len(raw) - 1 and "\\boxed" not in piece)
        if out and mergeable:
            out[-1] = (out[-1][0], b)
        else:
            out.append((a, b))
    return len(out)


def tokens_to_answer(tokens):
    run = ""
    for i, t in enumerate(tokens):
        run += t
        if "\\boxed{" in run:
            return i + 1
    return None

## The sweep

In [ ]:
RUN_ID = f"exp3-n{N_ITEMS}-k{FEW_SHOT_K}-cap{GEN_CAP}-{SEED}"
STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("tag", "dataset", "condition", "problem_id"))
STORE.report()

device = get_device()
DTYPE = harness.pick_dtype(device)


def sweep_model(entry):
    loaded = harness.load_model(entry["id"], device=device, dtype=DTYPE,
                                load_in_8bit=entry["quant"] == "8bit")
    model, tok = loaded["model"], loaded["tokenizer"]
    ctx = getattr(model.config, "max_position_embeddings", 32768)
    print(f"      context {ctx} tokens")

    for ds_name, ds in DATASETS.items():
        if ds_name in entry.get("skip_datasets", []):
            continue
        for cond in CONDITIONS:
            todo = [r for r in ds["items"]
                    if STORE.get(entry["tag"], ds_name, cond, r["unique_id"]) is None]
            bar = tqdm(ds["items"], desc=f"{entry['tag'][:22]:<22} {ds_name:<10} {cond:<9}",
                       unit="item")
            for row in bar:
                if STORE.get(entry["tag"], ds_name, cond, row["unique_id"]) is not None:
                    continue
                text = build_prompt(cond, row["problem"], ds["exemplars"])
                if entry["no_think"]:
                    text += " /no_think"     # Qwen3's in-prompt route to thinking off
                chat = tok.apply_chat_template([{"role": "user", "content": text}],
                                               tokenize=False, add_generation_prompt=True)
                n_prompt = len(tok(chat, add_special_tokens=False).input_ids)
                cap = max(64, min(GEN_CAP, ctx - n_prompt - 32))
                g = harness.complete(chat, model, tok, stop=["\nQuestion:"], max_new_tokens=cap)
                got = extract_final_candidate(g["text"])
                STORE.append({
                    "tag": entry["tag"], "model_id": entry["id"],
                    "quant": entry["quant"],
                    "dataset": ds_name, "condition": cond,
                    "problem_id": row["unique_id"], "question": row["problem"],
                    "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
                    "prompt_tokens": n_prompt, "cap": cap, "truncated": g["truncated"],
                    "tokens_generated": g["tokens_generated"],
                    "steps": n_steps(g["text"]),
                    "tokens_to_answer": tokens_to_answer(g["tokens"]),
                    "extracted_answer": got,
                    "correct": bool(grade_answer(got, row["answer"])),
                })
    del model, loaded
    harness.free_memory()


for entry in ROSTER:
    sweep_model(entry)
print("\nscreen complete")

## The tables

Everything below is written to `tables.md` as well as printed, so it can go straight into a
document without being retyped. Intervals are 95% Wilson; paired differences carry an exact
McNemar test, which uses only the items where the two conditions disagree.

In [ ]:
from math import comb

records = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
records = [r for r in records if r["run_id"] == RUN_ID]
B = "\\boxed"


def by_item(tag, ds, cond):
    return {r["problem_id"]: r for r in records
            if r["tag"] == tag and r["dataset"] == ds and r["condition"] == cond}


def mcnemar(a, b):
    """Exact two-sided test on the items where a and b disagree. Returns (a_only, b_only, p)."""
    both = set(a) & set(b)
    x = sum(1 for p in both if a[p]["correct"] and not b[p]["correct"])
    y = sum(1 for p in both if b[p]["correct"] and not a[p]["correct"])
    n = x + y
    if n == 0:
        return x, y, 1.0
    k = max(x, y)
    tail = sum(comb(n, i) for i in range(k, n + 1)) / 2 ** n
    return x, y, min(1.0, 2 * tail)


def cell(tag, ds, cond):
    rs = list(by_item(tag, ds, cond).values())
    if not rs:
        return None
    k = sum(r["correct"] for r in rs)
    lo, hi = wilson(k, len(rs))
    return {"tag": tag, "dataset": ds, "condition": cond, "n": len(rs),
            "accuracy": k / len(rs), "ci_lo": lo, "ci_hi": hi,
            "chain_share": mean([r["steps"] >= MIN_CHAIN_STEPS for r in rs]),
            "complies": mean([r["tokens_generated"] <= COMPLIES_UNDER for r in rs]),
            "boxed": mean([B in r["trace"] for r in rs]),
            "truncated": mean([r["truncated"] for r in rs]),
            "median_tokens": median([r["tokens_generated"] for r in rs]),
            "median_steps": median([r["steps"] for r in rs])}


TABLE = [c for e in ROSTER for ds in DATASETS for cond in CONDITIONS
         if (c := cell(e["tag"], ds, cond))]
LABEL = {"forced_blurt": "forced blurt", "bare": "bare", "zero_shot_cot": "+ step by step",
         "few_shot": "+ worked examples"}
MD = []


def emit(title, header, rows, note=None):
    MD.append(f"\n### {title}\n")
    MD.append("| " + " | ".join(header) + " |")
    MD.append("|" + "|".join("---" for _ in header) + "|")
    for r in rows:
        MD.append("| " + " | ".join(str(x) for x in r) + " |")
    if note:
        MD.append(f"\n{note}")
    print(f"\n### {title}")
    w = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    print("  " + "  ".join(str(h).ljust(w[i]) for i, h in enumerate(header)))
    for r in rows:
        print("  " + "  ".join(str(c).ljust(w[i]) for i, c in enumerate(r)))
    if note:
        print(f"  {note}")


# --- Table 1: the ladder -------------------------------------------------
for ds in DATASETS:
    rows = []
    for e in ROSTER:
        for cond in CONDITIONS:
            c = next((x for x in TABLE if x["tag"] == e["tag"] and x["dataset"] == ds
                      and x["condition"] == cond), None)
            if c:
                rows.append([e["tag"], LABEL[cond], f"{c['accuracy']:.0%}",
                             f"{c['ci_lo']:.0%}-{c['ci_hi']:.0%}", f"{c['median_tokens']:.0f}",
                             f"{c['median_steps']:.0f}", f"{c['chain_share']:.0%}"])
    emit(f"Table 1 — the prompt ladder on {ds}",
         ["model", "prompt", "solved", "95% CI", "tokens", "steps", "chain"], rows,
         f"n = {N_ITEMS} paired items per cell.")

In [ ]:
# --- Table 2: the contrasts that matter, with significance ---------------
CONTRASTS = [("forced_blurt", "bare", "cost of suppressing reasoning"),
             ("bare", "zero_shot_cot", "gain from asking for reasoning"),
             ("bare", "few_shot", "gain from showing an example")]
rows, SIG = [], []
for ds in DATASETS:
    for e in ROSTER:
        for lo_c, hi_c, what in CONTRASTS:
            a, b = by_item(e["tag"], ds, hi_c), by_item(e["tag"], ds, lo_c)
            if not a or not b:
                continue
            x, y, p = mcnemar(a, b)
            d = (mean([r["correct"] for r in a.values()])
                 - mean([r["correct"] for r in b.values()])) * 100
            verdict = "significant" if p < 0.05 else "not resolved"
            rows.append([e["tag"], ds, what, f"{d:+.0f}", f"{x}/{y}", f"{p:.3f}", verdict])
            SIG.append(dict(tag=e["tag"], dataset=ds, contrast=what, delta_pts=d,
                            discordant_hi=x, discordant_lo=y, p=p, significant=p < 0.05))
emit("Table 2 — what each prompt change is worth",
     ["model", "dataset", "change", "delta", "flips +/-", "p", "verdict"], rows,
     "Exact McNemar on paired items. 'flips' counts problems fixed / broken by the change.")

# --- Table 3: format and behaviour ---------------------------------------
rows = []
for ds in DATASETS:
    for e in ROSTER:
        for cond in CONDITIONS:
            c = next((x for x in TABLE if x["tag"] == e["tag"] and x["dataset"] == ds
                      and x["condition"] == cond), None)
            if c:
                rows.append([e["tag"], ds, LABEL[cond], f"{c['boxed']:.0%}",
                             f"{c['truncated']:.0%}", f"{c['complies']:.0%}"])
emit("Table 3 — format compliance and instruction following",
     ["model", "dataset", "prompt", "boxed", "truncated", "answered in <=40 tok"], rows,
     "'boxed' is the share using the answer format the stem asks for; unboxed answers "
     "fall back to the last number in the trace and are scored less reliably.")

# --- Table 4: which band has headroom ------------------------------------
rows = []
for e in ROSTER:
    for ds in DATASETS:
        ba = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "bare"), None)
        fs = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "few_shot"), None)
        fb = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "forced_blurt"), None)
        if ba and fs and fb:
            rows.append([e["tag"], ds, f"{ba['accuracy']:.0%}",
                         f"{100 - ba['accuracy'] * 100:.0f} pts",
                         f"{(fs['accuracy'] - ba['accuracy']) * 100:+.0f}",
                         f"{(ba['accuracy'] - fb['accuracy']) * 100:+.0f}"])
emit("Table 4 — headroom, and what it buys",
     ["model", "dataset", "bare", "room above bare", "few-shot gain", "suppression cost"], rows,
     "Experiment 2 found the few-shot gain correlates -0.79 with bare accuracy: prompting "
     "cannot move a model that is already near the ceiling.")

In [ ]:
import matplotlib.pyplot as plt

INK = {"forced_blurt": "#b23", "bare": "#9aa0ab", "zero_shot_cot": "#3b4a6b",
       "few_shot": "#a2704a"}
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.4 * len(DATASETS), 4.4), sharey=True)
axes = [axes] if len(DATASETS) == 1 else list(axes)
for ax, ds in zip(axes, DATASETS):
    tags = [e["tag"] for e in ROSTER]
    w = 0.8 / len(CONDITIONS)
    for j, cond in enumerate(CONDITIONS):
        # only cells that were actually run: a missing pass must leave a gap, not a
        # zero bar, which a reader would take for a score of nought
        found = [(i + j * w - 0.4, c) for i, t in enumerate(tags)
                 if (c := next((x for x in TABLE if x["tag"] == t and x["dataset"] == ds
                                and x["condition"] == cond), None))]
        if not found:
            continue
        xs = [p for p, _ in found]
        cs = [c for _, c in found]
        ax.bar(xs, [c["accuracy"] * 100 for c in cs], width=w,
               color=INK[cond], label=LABEL[cond] if ax is axes[0] else None)
        ax.errorbar(xs, [c["accuracy"] * 100 for c in cs],
                    yerr=[[(c["accuracy"] - c["ci_lo"]) * 100 for c in cs],
                          [(c["ci_hi"] - c["accuracy"]) * 100 for c in cs]],
                    fmt="none", ecolor="#41474f", elinewidth=1, capsize=2)
    ax.set_xticks(range(len(tags)))
    ax.set_xticklabels(tags)
    ax.set_ylim(0, 100)
    ax.set_title(ds, fontsize=9, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("solved (%)")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(f"the prompt ladder, n={N_ITEMS} paired items, 95% Wilson",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_experiment_3.png", dpi=160, bbox_inches="tight")
plt.show()

## The record

In [ ]:
derived = {
    "manifest": {
        "notebook": "experiment_3_prompt_ladder_at_n100.ipynb", "run_id": RUN_ID,
        "harness": harness.fingerprint(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "notebook_git_sha": harness.git_sha(),
        "n_items": N_ITEMS, "few_shot_k": FEW_SHOT_K, "gen_cap": GEN_CAP, "seed": SEED,
        "conditions": CONDITIONS, "roster": ROSTER,
        "datasets": {k: {"hash": v["hash"], "n": len(v["items"]),
                         "exemplars": [e["unique_id"] for e in v["exemplars"]]}
                     for k, v in DATASETS.items()},
        "versions": {"torch": torch.__version__,
                     "transformers": __import__("transformers").__version__,
                     "device": str(device)},
    },
    "grid": TABLE,
    "contrasts": SIG,
    "thresholds": {"complies_under_tokens": COMPLIES_UNDER, "min_chain_steps": MIN_CHAIN_STEPS},
}
DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
TABLES_PATH.write_text(
    f"# Experiment 3 — the prompt ladder\n\n"
    f"`{RUN_ID}` · n={N_ITEMS} paired items · "
    f"{datetime.now(timezone.utc).isoformat(timespec='minutes')}\n"
    + "\n".join(MD) + "\n", encoding="utf-8")

print(f"\nderived.json   {harness.kb(DERIVED_PATH):>9.1f} KB")
print(f"tables.md      {harness.kb(TABLES_PATH):>9.1f} KB   <- paste-ready")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB")
n_sig = sum(1 for s in SIG if s["significant"])
print(f"\n{n_sig} of {len(SIG)} prompt contrasts resolved at p<0.05")